### Patch Antenna Meshing
We now apply a refined mesh for the patch antenna. Because the electromagnetic fields change rapidly near the metallic patch and the substrate edges, we increase the mesh density in these areas to ensure high simulation accuracy for resonance and radiation patterns.

In [1]:
import gmsh
import math
import os
import json

from palacetoolkit.viz import view_mesh
from palacetoolkit.geometry import extract_tag, xmin, xmax, ymin, ymax, zmin, zmax
from palacetoolkit.mesh import refine_near_surfaces, refine_surfaces


### Parameters:
- l : Patch length along x-axis, specified as a scalar in meters.
- w : Patch width along y-axis, specified as a scalar in meters.
- h : Patch height along z-axis, specified as a scalar in meters. 
- l1 : Ground plane length along x-axis, specified as a scalar in meters
- w1 : Ground plane width along y-axis, specified as a scalar in meters
- l2 : Notch length along x-axis, specified as a scalar in meters. 
- w2 : Notch width along x-axis, specified as a scalar in meters. 
- w3 : Strip line width along y-axis, specified as a scalar in meters.
- air_height : Air box height along z-axis, specified as a scalar in meters.  
- air_margin : Air box margin along x and y axes, specified as a scalar in meters.
- freq  : Simulation frequency in GHz, specified as a scalar.
- filename : Output mesh filename, specified as a string.

In [2]:
l: float = 0.030
w: float = 0.029
l1: float = 0.06
w1: float = 0.06
l2: float = 0.008
w2: float = 0.003
w3: float = 0.001
h: float = 0.0013
air_height: float = 0.025    
air_margin: float = 0.025    
freq: float = 3.3
filename: str = "patch_antenna.msh"

### Initialize the model

In [3]:
gmsh.initialize()
gmsh.model.add("patch_antenna")
kernel = gmsh.model.occ

### Geometry Construction
In this step, we build the physical structure of the patch antenna. We define the substrate, the metallic ground plane, and the antenna patch itself.

Patch Design: We combine the main patch and the feed line, including an inset to help with impedance matching.
Excitation: A lumped port is created and positioned to feed the antenna.
Domain Setup: We enclose the entire structure in an "air box," which defines the simulation region (the radiation space).
Finalizing: We use boolean operations (fragmenting) to ensure all components are properly connected and recognized as a single cohesive model by the solver.

In [4]:
# Total domain bounds
total_xmin = -l1/2 - air_margin
total_xmax = l1/2 + air_margin
total_ymin = -w1/2 - air_margin
total_ymax = w1/2 + air_margin
total_zmax = h + air_height

substrate = kernel.addBox(-l1/2, -w1/2, 0, l1, w1, h)

ground_plane = kernel.addRectangle(-l1/2, -w1/2, 0, l1, w1)

main_patch = kernel.addRectangle(-l/2, -w/2, h, l, w)
inset = kernel.addRectangle(-l/2, -w2/2, h, l2, w2)

patch_dimtags, _ = kernel.cut(
    [(2, main_patch)], 
    [(2, inset)], 
    removeObject=True, removeTool=True
)
kernel.synchronize()


feed_length = (l1 - l)/2 + l2
feed_line = kernel.addRectangle(-l1/2, -w3/2, h, feed_length, w3)

top_conductor, _ = kernel.fuse(
    patch_dimtags, [(2, feed_line)],
    removeObject=True, removeTool=True
)
kernel.synchronize()


gap = 0
lumped_port = kernel.addRectangle(-l1/2 + gap, -w3/2, 0, h - gap, w3)
kernel.rotate([(2, lumped_port)], -l1/2, 0, 0, 0, 1, 0, -math.pi/2)
kernel.synchronize()

air_box = kernel.addBox(
    total_xmin, total_ymin, 0,
    total_xmax - total_xmin,
    total_ymax - total_ymin,
    total_zmax
)
kernel.synchronize()

all_tools = [(2, ground_plane)] + top_conductor + [(2, lumped_port)]

result, result_map = kernel.fragment(
    [(3, substrate), (3, air_box)], 
    all_tools
)
kernel.synchronize()
kernel.removeAllDuplicates()
kernel.synchronize()

Info    : Cannot bind existing OpenCASCADE surface 8 to second tag 9
Info    : Could not preserve tag of 2D object 9 (->8)


### Defining Physical Groups
We categorize the geometry parts by checking their coordinates. This automatically groups the antenna components (patch, ground, port) and surrounding regions (substrate, air) into "Physical Groups," which are essential for defining simulation boundaries and material properties in Palace.

In [5]:
# regions
all_2d = gmsh.model.getEntities(2)
all_3d = gmsh.model.getEntities(3)

tol = 1e-6

ground_tags = []
sides = []
patch_tags = []
port_tags = []
farfield_tags = []
substrate_tags = []
air_tags = []

# 2D surfaces
for dim, tag in all_2d:
    bbox = gmsh.model.getBoundingBox(dim, tag)
    xmin, ymin, zmin = bbox[0], bbox[1], bbox[2]
    xmax, ymax, zmax = bbox[3], bbox[4], bbox[5]
    
    # Ground plane
    if (math.isclose(zmin, 0, abs_tol=tol) and 
        math.isclose(zmax, 0, abs_tol=tol) and
        xmin >= -l1/2 - tol and xmax <= l1/2 + tol):
        ground_tags.append(tag)
        
    # Patch + feed
    elif (math.isclose(zmin, h, abs_tol=tol) and 
            math.isclose(zmax, h, abs_tol=tol) and
            math.isclose(xmax, l/2, abs_tol=tol)):
        patch_tags.append(tag)
        
    # Lumped port
    elif (math.isclose(xmin, -l1/2, abs_tol=tol) and 
            math.isclose(xmax, -l1/2, abs_tol=tol) and
            math.isclose(ymin, -w3/2, abs_tol=tol) and
            math.isclose(ymax, w3/2, abs_tol=tol) and
            zmin >= gap - tol and zmax <= h + tol):
        port_tags.append(tag)
        
    # Far-field (outer air box surfaces)
    elif (math.isclose(xmin, -l1/2 - air_margin, abs_tol=tol) or
            math.isclose(xmax, l1/2 + air_margin, abs_tol=tol) or
            math.isclose(ymin, -w1/2 - air_margin, abs_tol=tol) or
            math.isclose(ymax, w1/2 + air_margin, abs_tol=tol) or
            math.isclose(zmax, h + air_height, abs_tol=tol)):
        farfield_tags.append(tag)
    else:
        sides.append(tag)

# 3D volumes
for dim, tag in all_3d:
    bbox = gmsh.model.getBoundingBox(dim, tag)
    zmax = bbox[5]
    
    if math.isclose(zmax, h, abs_tol=tol):
        substrate_tags.append(tag)
    else:
        air_tags.append(tag)

# physical groups
pg_map = {
}

if substrate_tags:
    pg_map["substrate"] = gmsh.model.addPhysicalGroup(3, substrate_tags, name = "Substrate")
if air_tags:
    pg_map["air"] = gmsh.model.addPhysicalGroup(3, air_tags, name = "Air")
if ground_tags:
    pg_map["ground_plane"] = gmsh.model.addPhysicalGroup(2, ground_tags, name = "GroundPlane")
if patch_tags:
    pg_map["patch"] = gmsh.model.addPhysicalGroup(2, patch_tags, name = "Patch")
if port_tags:
    pg_map["lumped_port"] = gmsh.model.addPhysicalGroup(2, port_tags, name = "LumpedPort")
if farfield_tags:
    pg_map["farfield"] = gmsh.model.addPhysicalGroup(2, farfield_tags, name = "FarField")
if sides:
    pg_map["sides"] = gmsh.model.addPhysicalGroup(2, sides, name = "Sides")

### Mesh generation and refinement

In [6]:
wavelength = 3e8 / (freq * 1e9)
mesh_size = wavelength / 15

refine_surfaces(port_tags, mesh_size) 

gmsh.model.mesh.generate(3)
gmsh.model.mesh.setOrder(1)

gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
gmsh.option.setNumber("Mesh.Binary", 0)
gmsh.write(filename)

gmsh.fltk.run()
gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 45 (Line)
Info    : [ 10%] Meshing curve 46 (Line)
Info    : [ 10%] Meshing curve 47 (Line)
Info    : [ 10%] Meshing curve 48 (Line)
Info    : [ 10%] Meshing curve 49 (Line)
Info    : [ 20%] Meshing curve 50 (Line)
Info    : [ 20%] Meshing curve 51 (Line)
Info    : [ 20%] Meshing curve 52 (Line)
Info    : [ 20%] Meshing curve 53 (Line)
Info    : [ 30%] Meshing curve 54 (Line)
Info    : [ 30%] Meshing curve 55 (Line)
Info    : [ 30%] Meshing curve 56 (Line)
Info    : [ 30%] Meshing curve 57 (Line)
Info    : [ 40%] Meshing curve 58 (Line)
Info    : [ 40%] Meshing curve 59 (Line)
Info    : [ 40%] Meshing curve 60 (Line)
Info    : [ 40%] Meshing curve 61 (Line)
Info    : [ 50%] Meshing curve 62 (Line)
Info    : [ 50%] Meshing curve 63 (Line)
Info    : [ 50%] Meshing curve 64 (Line)
Info    : [ 50%] Meshing curve 65 (Line)
Info    : [ 60%] Meshing curve 66 (Line)
Info    : [ 60%] Meshing curve 67 (Line)
Info    : [ 60%] Meshing curve 68

### Simulation Configuration
We define the key parameters for the electromagnetic simulation here. These settings control the frequency sweep range, material properties (dielectric constant and loss tangent for the substrate), and solver-specific configurations like port impedance and mesh order.

- output_file  : output filename for the configuration JSON file
- freq_min : minimum frequency for the simulation (GHz)  
- freq_max: maximum frequency for the simulation (GHz)
- freq_step: frequency step for the simulation (GHz)
- eps_r: relative permittivity of the substrate
- loss_tan: loss tangent of the substrate
- port_impedance: characteristic impedance of the lumped port (Ohms)
- solver_order: order of the finite element basis functions for the simulation (e.g., 1 for linear, 2 for quadratic)

In [7]:
output_file: str = "patch_antenna.json"
freq_min: float = 3.0
freq_max: float = 3.5
freq_step: float = 0.005
eps_r: float = 2.2
loss_tan: float = 0.0009
port_impedance: float = 50.0
solver_order: int = 2

### Generating the Palace Configuration File
Finally, we assemble the simulation parameters into a JSON configuration .

In [8]:
def attr(name):
        return [pg_map[name]] if name in pg_map else []

config = {
    "Problem": {
        "Type": "Driven",
        "Verbose": 2,
        "Output": "/work/results/patch_antenna/"
    },

    "Model": {
        "Mesh": f"/work/{filename}",
        "L0": 1.0,
        "Refinement": {}
    },

    "Domains": {
        "Materials": [
            {
                "Attributes": attr("substrate"),
                "Permittivity": eps_r,
                "Permeability": 1.0,
                "LossTan": loss_tan
            },
            {
                "Attributes": attr("air"),
                "Permittivity": 1.0,
                "Permeability": 1.0
            }
        ]
    },

    "Boundaries": {
        "PEC": {
            "Attributes": attr("ground_plane") + attr("patch")
        },

        "LumpedPort": [
            {
                "Index": 1,
                "Attributes": attr("lumped_port"),
                "R": port_impedance,
                "Excitation": True,
                "Direction": "+Z"
            }
        ],

        "Absorbing": {
            "Attributes": attr("farfield"),
            "Order": 1
        }
    },

    "Solver": {
        "Order": solver_order,
        "Device": "CPU",

        "Driven": {
            "MinFreq": freq_min,
            "MaxFreq": freq_max,
            "FreqStep": freq_step,
            "AdaptiveTol": 0.001
        },

        "Linear": {
            "Type": "Default",
            "KSPType": "GMRES",
            "Tol": 1.0e-8,
            "MaxIts": 200,
            "ComplexCoarseSolve": True
        }
    }
}

script_dir = os.getcwd()
config_path = os.path.join(script_dir, output_file)
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Palace config written to {config_path}")

Palace config written to /home/martin/Desktop/PalaceToolkit/docs/examples/patch_antenna.json
